# AI Engineering Challenge: valmis RAG-ratkaisu

Ratkaisu hakee kysymykseen sopivat Wikipedia-katkelmat ja antaa niiden perusteella lyhyen vastauksen lähdeviitteineen. Mallina toimii oletuksena paikallinen **llama3.2:3b**.

**Aja solut ylhäältä alas**, tai valitse **Run → Run All Cells**. Ollaman pitää olla käynnissä. Kehitysjoukon ja lopullisen arvioinnin malliajot voivat kestää useita minuutteja.

Alkuperäinen ratkaisu on säilytetty tiedostossa `rag/baseline.py`. Arviointikoodia `eval/` ei ole muutettu. Mitatut vertailut ja ajokomentojen ohjeet ovat repon `WORKSHOP_RESULTS.md`-tiedostossa.


## Valmis vertailutulos, 23.9.2026

80 kysymyksen lopputesti, sama `llama3.2:3b`-malli ja muuttamaton arviointikoodi:

| Mittari | Alkuperäinen | Parannettu |
|---|---:|---:|
| Pisteet / 100 | 16,01 | **71,71** |
| Täsmälleen oikeat vastaukset | 6/80 | 49/80 |
| Tokenit | 16 127 | 51 351 |
| Suoritusvirheet | 0 | 0 |

Laatu parani, mutta tokenkulutus kasvoi noin 3,2-kertaiseksi. Vertailu ajettiin
GitHubissa tämän notebookin käyttämällä vastausfunktiolla. Koneesi ajoaika ja
uudelleenajon tulos voivat poiketa näistä.

[Lopputestin lokit ja tulokset](https://github.com/nexpertfinland/ai-engineering-challenge/actions/runs/35866150659)


## 1. Lataa aineisto ja tarkista käytettävä malli

In [ ]:
import sys, pathlib, json

def find_root(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError("Could not find project root (pyproject.toml not found)")

ROOT = find_root(pathlib.Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag.llm import PROVIDER, DEFAULT_MODEL, ensure_model_ready

CORPUS_PATH = ROOT / "data" / "corpus.json"
DEV_QA_PATH = ROOT / "data" / "dev_qa.json"
TEST_QA_PATH = ROOT / "eval" / "test_qa.json"

if CORPUS_PATH.exists() and DEV_QA_PATH.exists() and TEST_QA_PATH.exists():
    corpus = json.loads(CORPUS_PATH.read_text())
    dev_qa = json.loads(DEV_QA_PATH.read_text())
else:
    # First run: no local dataset yet (it's gitignored, never pushed to the repo).
    # Build it from SQuAD and cache it locally so later runs are instant. The split
    # is deterministic (fixed seed), so eval/test_qa.json comes out identical for
    # every participant without ever being committed either.
    print("No local dataset found — downloading SQuAD and building the corpus (one-time, needs internet)...")
    from rag.dataset import build_splits

    corpus, dev_qa, test_qa = build_splits()
    CORPUS_PATH.parent.mkdir(exist_ok=True)
    TEST_QA_PATH.parent.mkdir(exist_ok=True)
    CORPUS_PATH.write_text(json.dumps(corpus, indent=2))
    DEV_QA_PATH.write_text(json.dumps(dev_qa, indent=2))
    TEST_QA_PATH.write_text(json.dumps(test_qa, indent=2))

print(f"LLM provider: {PROVIDER}  |  model: {DEFAULT_MODEL}")
print(f"Corpus: {len(corpus)} articles, {sum(len(d['text']) for d in corpus):,} chars total")
print(f"Dev QA: {len(dev_qa)} questions (visible, gold answers included)")


## 2. Hae vastausta tukeva teksti

Alkuperäinen haku katkoo tekstin 500 merkin paloiksi ja laskee yhteisten sanojen määriä. Parannettu haku säilyttää kappaleet, limittää pitkät katkelmat ja yhdistää **BM25:n** ja **TF-IDF:n**. Harvinaiset, kysymystä erottelevat sanat saavat enemmän painoa.

Alla verrataan vain tiedonhakua 40 näkyvällä kehityskysymyksellä. `answer_context_recall` kertoo, kuinka usein oikean artikkelin mukana tulleessa katkelmassa on jokin hyväksytty vastaus. **Tämä ei ole lopullinen tehtäväpistemäärä.**


In [ ]:
from rag.baseline import BaselinePipeline
from rag.pipeline import RAGPipeline
from rag.evaluation import retrieval_report, evaluate_dev

baseline = BaselinePipeline(corpus)
pipeline = RAGPipeline(corpus, top_k=3)
answer_question = pipeline.answer_question

print(f"Parannettu haku: {len(pipeline.retriever.chunks)} katkelmaa")
print("Alkuperäinen haku (1 katkelma):")
print(json.dumps(retrieval_report(baseline.retrieve, dev_qa, k=1), indent=2))
print("Parannettu haku (3 katkelmaa):")
print(json.dumps(retrieval_report(pipeline.retrieve, dev_qa, k=3), indent=2))


## 3. Muodosta lyhyt vastaus ja tarkista lähde

Malli saa kolme parhaiten sopivaa katkelmaa. Se palauttaa lyhyen vastauksen ja sitä tukevan katkelman numeron. Koodi hyväksyy vastauksen vain, jos se löytyy ilmoitetusta katkelmasta. Lähteeksi merkitään vain vastauksen tukeva artikkeli.

Jos malli ei löydä vastausta tai palauttaa virheellisen rakenteen, tuloksena on tyhjä vastaus. Vastauksia ei arvata. Jokaisen mallikutsun tokenit lasketaan mukaan, myös hylättyjen vastausten.


In [ ]:
ensure_model_ready()

sample = dev_qa[0]
result = answer_question(sample["question"])
print("Kysymys:", sample["question"])
print("Hyväksytyt vastaukset (vain kehitysjoukko):", sample["answers"])
print("Ratkaisun vastaus:", result)


## 4. Kehitysjoukon tulos

Tässä saa tarkastella virheitä ja kokeilla parannuksia. Ajossa on 40 kysymystä. Yksi mallikutsu tehdään kutakin kysymystä kohti, jos tiedonhaku löytää katkelmia.


In [ ]:
dev_results = evaluate_dev(answer_question, dev_qa)
print(json.dumps({k: v for k, v in dev_results.items() if k != "details"}, indent=2))


## 5. Lopullinen arviointi: 80 erillistä kysymystä

Alla on alkuperäinen lukittu arviointifunktio. Testijoukon vastauksia ei käytetä haussa, vastausten muodostamisessa tai asetusten valinnassa.

**Laatupisteet:** `100 × (0.8 × F1 + 0.2 × lähdeosumat)`.
**Tokenkulutus:** erillinen mittari, jossa pienempi on parempi. Useampi lähdekatkelma voi parantaa laatua mutta kasvattaa syötteen tokenmäärää.


In [ ]:
from eval.harness import run_hidden_eval

results = run_hidden_eval(answer_question)
results


## Tuloksen tulkinta

- `score`: tehtävän laatupisteet, suurempi on parempi.
- `f1`: vastauksen sanojen vastaavuus hyväksyttyihin vastauksiin.
- `citation_hit_rate`: oikeaan artikkeliin osuvien lähdeviitteiden osuus.
- `total_tokens`: kaikki arviointiajon syöte- ja vastaustokenit.
- `errors`: suorituksen virheet. Tavoite on nolla.
- `total_time_sec`: koko ajon kesto tällä koneella.

Alkuperäisen ja parannetun ratkaisun täydellisen vertailun voi ajaa PowerShellissa komennolla `uv run python scripts/benchmark.py --split hidden --pipeline both`. Ajot tallentuvat `results/`-kansioon. Säilytä myös mallin nimi ja ajon päivämäärä tuloksen yhteydessä.
